# Monitoring a Slurm cluster — hands-on

**Run the commands, read the output, learn what the scheduler knows.**

This notebook is a guided tour of the Slurm commands a cluster user or
administrator runs every day. Each section introduces one command, explains
what it answers, and then runs a small script that puts it to work. The
commands are real Slurm syntax — only the cluster is fictional.

> **The fake cluster.** `fake_slurm/` simulates **kapoorlabs-fr**, a small
> French HPC with three GPU partitions (`h100`, `h200`, `b200`) and one CPU
> partition. `sinfo`, `squeue`, `scontrol`, `sstat`, `sacct` and
> `sacctmgr` all answer from it.
>
> Run **Setup** once, then work top to bottom. On a real login node, skip
> Setup — the remaining cells run unchanged against your cluster.

## What you will learn

| # | Command | Question it answers |
|:--:|---|---|
| 1 | `sinfo` | What hardware exists, and how much of it is free? |
| 2 | `srun` | How do I land a shell on a free node? |
| 3 | `scontrol show node` | What has the scheduler promised each node? |
| 4 | `squeue` | What is running now, and what is waiting? |
| 5 | `scontrol show job` | What exactly did a job ask for? |
| 6 | `sstat` | How much memory is a job using right now? |
| 7 | `sacct` | What happened to jobs that already finished? |
| 8 | `sacctmgr` | What am I allowed to run? |
| 9 | — | The one-page cheat sheet |
| 10 | `hpc_dashboard.py` | Can I watch all of this live? |

**By the end** you will have built four small monitoring scripts —
`hpc_info.py`, `show_free_nodes.py`, `scontroller_nodes.py` and
`slurm_job_checker.py`. Each is just a scheduled call to the commands above,
with the output parsed into a table you can read at a glance.

The full command reference is
[`SLURM_TUTORIAL.md`](../../SLURM_TUTORIAL.md) at the repository root.

---

## Setup · point the commands at the fake cluster

Slurm commands are found on `PATH`, and they read the cluster's contract
from `SLURM_CONF`. Putting `fake_slurm/` first on `PATH` makes `sinfo` in
this shell the fake one; pointing `SLURM_CONF` at its `slurm.conf` gives it
the right cluster name and limits.

> **On a real cluster:** skip this cell. `sinfo` is already on your `PATH`
> and `SLURM_CONF` points at `/etc/slurm/slurm.conf`.
>
> The same thing in a terminal:
>
> ```bash
> export PATH="$PWD/fake_slurm:$PATH"
> export SLURM_CONF="$PWD/fake_slurm/slurm.conf"
> ```

The cell below prints where each command now resolves, then asks the fake
controller to identify itself.

In [1]:
import os
import shutil
from pathlib import Path

fake = Path("fake_slurm").resolve()
assert (fake / "sinfo").exists(), "run this notebook from tutorials/hpc_tutorials/"

# Commands resolve fake_slurm/ first; the fake reads the same slurm.conf
# a real cluster would use.
os.environ["PATH"] = f"{fake}{os.pathsep}{os.environ['PATH']}"
os.environ["SLURM_CONF"] = str(fake / "slurm.conf")

print(f"config    : {fake / 'slurm.conf'}")
print(f"sinfo     : {shutil.which('sinfo')}")
print(f"SLURM_CONF: {os.environ['SLURM_CONF']}")
!sinfo -h

config    : /Users/vkapoor/python_workspace/LiveTutorials/tutorials/hpc_tutorials/fake_slurm/slurm.conf
sinfo     : /Users/vkapoor/python_workspace/LiveTutorials/tutorials/hpc_tutorials/fake_slurm/sinfo
SLURM_CONF: /Users/vkapoor/python_workspace/LiveTutorials/tutorials/hpc_tutorials/fake_slurm/slurm.conf
slurm 23.02.4


## 1 · `sinfo` — what the cluster has

`sinfo` asks the controller about **nodes and partitions**: how many CPUs,
how much memory, which GPUs, what wall-time limit. `-N` asks for one line
per node, and `-o` chooses the fields:

```bash
sinfo -N -o "%N|%P|%C|%m|%e|%G|%l|%T"
```

Reading the field codes is most of the job:

| Code | Field | Meaning |
|:--:|---|---|
| `%N` | node | node name |
| `%P` | partition | queue name; `*` marks the default partition |
| `%C` | CPUs | `allocated/idle/other/total` |
| `%m` | memory | **configured** RAM, in MB |
| `%e` | free memory | what the node reports as free right now, MB |
| `%G` | GRES | generic resources — here, the GPUs |
| `%l` | time limit | longest job the partition accepts |
| `%T` | state | `idle`, `mixed`, `allocated`, `drained`, `down`, … |

In [2]:
!sinfo -N -o "%N|%P|%C|%m|%e|%G|%l|%T"

h100-1|h100*|16/16/0/32|2048000|800000|gpu:h100.80gb:8(S:0-1)|7-00:00:00|mixed
h100-2|h100*|32/0/0/32|2048000|400000|gpu:h100.80gb:8(S:0-1)|7-00:00:00|allocated*
h100-3|h100*|32/0/0/32|2048000|1900000|gpu:h100.80gb:8(S:0-1)|7-00:00:00|allocated
h100-4|h100*|0/32/0/32|2048000|2000000|gpu:h100.80gb:8(S:0-1)|7-00:00:00|idle
h200-1|h200*|24/24/0/48|3072000|500000|gpu:h200.141gb:8|7-00:00:00|mixed
h200-2|h200*|48/0/0/48|3072000|2900000|gpu:h200.141gb:8|7-00:00:00|allocated
h200-3|h200*|0/48/0/48|3072000|3050000|gpu:h200.141gb:8|7-00:00:00|idle
b200-1|b200*|32/32/0/64|4096000|100000|gpu:b200.192gb:8|14-00:00:00|mixed
b200-2|b200*|0/64/0/64|4096000|4000000|gpu:b200.192gb:8|14-00:00:00|drained
cpu-1|cpu*|64/64/0/128|1024000|300000|(null)|3-00:00:00|mixed
cpu-2|cpu*|0/128/0/128|1024000|1000000|(null)|3-00:00:00|idle


### How to read it

- **`%C` is four numbers**: allocated, idle, other, total. The middle number
  is what a job can actually get right now.
- **`mixed` means part of the node is free** — some CPUs allocated, some
  idle. `allocated` means no free CPUs; `idle` means nothing is running.
- **A node can print more than once** if it belongs to several partitions.
  Count unique node names, not rows — this matters when totalling GPUs.
- **`drained` / `down` nodes still show idle CPUs.** They are out of
  service: an administrator (or a failing health check) removed them.
  Never recommend one to a user.
- **`%e` is real free memory**, while `%m` is installed memory. The gap is
  not automatically "leaked": Linux uses spare RAM for page cache and
  reclaims it when jobs need it.

> **GRES strings are packed.** `gpu:h100.80gb:8(S:0-1)` means 8 × H100
> 80 GB, with a CPU-affinity suffix. Parse with a regex that stops before
> `(S:…`, and count GPUs on **unique** nodes.

### Putting it together — `hpc_info.py`

Raw `sinfo` is one line per node. The first script rolls it up into the
view you actually want when you arrive at a new cluster:

- one row per **partition** — nodes, CPUs, GPU types and counts, memory,
  time limit;
- **cluster-wide GPU totals** counted over unique nodes, so a node shared
  by two partitions is not counted twice;
- **your own access** (account, partitions, QOS) from `sacctmgr`;
- optionally quota and balance checks (`extras=true`) when the cluster
  provides `show-quota` / `sbalance`.

It runs `sinfo` twice — once per node, once for the partition summary — and
parses both. This is the same inventory the dashboard shows at the top.

In [3]:
!python hpc_info.py

  PARTITIONS
PARTITION  NODES  CPUS  GPUs            MEM(GB)  TIMELIMIT  
---------  -----  ----  --------------  -------  -----------
b200       2      128   16× B200.192GB  4000     14-00:00:00
cpu        2      256   -               1000     3-00:00:00 
h100       4      128   32× H100.80GB   2000     7-00:00:00 
h200       3      144   24× H200.141GB  3000     7-00:00:00 

Cluster GPU totals (unique nodes): 72 GPUs
    32× H100.80GB
    24× H200.141GB
    16× B200.192GB

Your Slurm access (user=vkapoor):
  account=kapoorlabs   partition=ALL            qos=normal


> **Look at the output.** 72 GPUs across 11 nodes — and the per-partition
> rows add up to exactly that, because no node here belongs to two
> partitions. On a cluster where nodes are shared between partitions, the
> row sum would over-count; the unique-node total would not.
>
> The `access` line comes from `sacctmgr`, not `sinfo`: it is what your
> jobs will be checked against.

## 2 · Finding free capacity — and landing on it

A user's real question is not "how many nodes are idle" but **"where can I
run right now?"** The second script answers it from one `sinfo` call:

- keeps nodes with **idle CPUs**;
- drops **drained / down / failed** nodes — free CPUs there are not usable;
- sorts fully `idle` nodes first, then by idle CPUs;
- splits GPU and CPU-only nodes, and prints a ready-to-paste `srun` command
  for the best node of each kind.

In [4]:
!python show_free_nodes.py


=== GPU NODES ===

Node         Idle/Total CPUs  GPU                  Memory     State     
----------------------------------------------------------------------
h200-3         48/48         h200.141gb x8        3072GB      idle      
h100-4         32/32         h100.80gb x8         2048GB      idle      
b200-1         32/64         b200.192gb x8        4096GB      mixed     
h200-1         24/48         h200.141gb x8        3072GB      mixed     
h100-1         16/32         h100.80gb x8         2048GB      mixed     

=== CPU-ONLY NODES ===

Node         Idle/Total CPUs  GPU                  Memory     State     
----------------------------------------------------------------------
cpu-2         128/128        -                    1024GB      idle      
cpu-1          64/128        -                    1024GB      mixed     

--- Quick commands ---
GPU: srun --nodelist=h200-3 --gres=gpu:1 --pty bash
CPU: srun --nodelist=cpu-2 --pty bash



### `srun` — act through the scheduler

`srun` asks the scheduler for an allocation and runs a command inside it.
With `--pty bash` you get an interactive shell on a compute node:

```bash
srun --nodelist=h200-3 --gres=gpu:1 --pty bash   # GPU node
srun --nodelist=cpu-2 --pty bash                 # CPU-only node
```

> **Use `srun`, not `ssh`.** Many clusters block direct SSH to compute
> nodes, and a job started outside the scheduler is invisible to it — no
> accounting, no cgroup limits, and no cleanup when you disconnect. `srun`
> goes through the controller, so the allocation is charged and released
> properly.

`--nodelist` pins a specific node because you already found a free one.
For normal work, omit it and let the scheduler choose:

```bash
srun --partition=h100 --gres=gpu:1 --pty bash
```

## 3 · `scontrol show node` — the scheduler's promise

`sinfo` is a summary; `scontrol show node` is the full record for one node,
including what the scheduler has **reserved** for running jobs.

```bash
scontrol show node h100-1
```

| Field | Meaning |
|---|---|
| `RealMemory` | RAM installed on the node, MB |
| `CPUAlloc` / `CPUTot` | CPUs allocated / installed |
| `CfgTRES` | resources **configured** on the node |
| `AllocTRES` | sum of every running job's request — the **promise** |
| `Gres` | GPU inventory and sharing layout |
| `State`, `Reason` | current state, and why if it is not serving |
| `Partitions` | which queues may place jobs here |

Start with a node that is in service:

In [5]:
!scontrol show node h100-1

NodeName=h100-1 Arch=x86_64 CoresPerSocket=32 CPUAlloc=16 CPUTot=32 RealMemory=2048000 State=mixed Partitions=h100 CfgTRES=cpu/32,mem/2048000M,billing/32 AllocTRES=cpu=16,mem=512G,gres/gpu=8 Gres=gpu:h100.80gb:8(S:0-1) AvailableFeatures=gpu,large-mem,x86_64 ActiveFeatures=gpu,large-mem


Now the same command on the node that is **not** serving. The difference
is the `Reason=` field — and it is the first thing to read when a user says
"my job won't start":

```bash
scontrol show node b200-2
```

In [6]:
!scontrol show node b200-2

NodeName=b200-2 Arch=x86_64 CoresPerSocket=32 CPUAlloc=0 CPUTot=64 RealMemory=4096000 State=drained Reason=firmware update [admin@2026-11-20] Partitions=b200 CfgTRES=cpu/64,mem/4096000M,billing/64 AllocTRES= Gres=gpu:b200.192gb:8 AvailableFeatures=gpu,large-mem,x86_64 ActiveFeatures=gpu,large-mem


### `AllocTRES` vs `RealMemory`

`AllocTRES=cpu=16,mem=512G,gres/gpu=8` does **not** mean the node is out of
memory. It means running jobs have *reserved* 512 GB out of 2 TB — and
reservations are often larger than what processes actually touch.

The node's own view (`sinfo %e`, `sstat`) is the second opinion. Comparing
them is how you find a node whose jobs are over-allocating.

### Putting it together — `scontroller_nodes.py`

The third script runs `scontrol show node -o` — one line per node,
parse-friendly — then prints each node's State, TRES, GRES and features,
classifies every node (free / busy / broken), and finishes with the resource
keys from `slurm.conf`:

- `SelectTypeParameters` — how consumable resources are counted;
- `TaskPlugin=task/cgroup` — whether limits are actually enforced;
- `AccountingStorageTRES` — whether memory or GPUs are measured at all.

In [7]:
!python scontroller_nodes.py

=== SLURM Cluster Node Status (11 nodes) ===



Node: h100-1
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: mixed
  Partitions: h100
  AllocTRES: cpu=16,mem=512G,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-2
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: allocated*
  Partitions: h100
  AllocTRES: cpu=32,mem=768G,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-3
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: allocated
  Partitions: h100
  AllocTRES: cpu=32,mem=1T,gres/gpu=8
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h100-4
  ActiveFeatures: gpu,large-mem
  AvailableFeatures: gpu,large-mem,x86_64
  Gres: gpu:h100.80gb:8(S:0-1)
  State: idle
  Partitions: h100
  AllocTRES: (none)
  CfgTRES: cpu/32,mem/2048000M,billing/32
  mem: 2048000

Node: h200-1
 

> **The output is long by design** — one block per node, then the cluster
> config. Read the summary line at the bottom first:
> `All-Free/Idle`, `Busy`, `Drained/Down`.

## 4 · `squeue` — what is running and what is waiting

`sinfo` describes capacity; `squeue` describes **jobs**. It prints running
and pending jobs, one per line, with the fields you select:

```bash
squeue -o "%i|%u|%j|%N|%P|%C|%m|%b|%M|%L"
```

| Code | Field | Meaning |
|:--:|---|---|
| `%i` | job id | array tasks append `_<task>`, e.g. `10020_74` |
| `%u`, `%j` | user, name | who submitted what |
| `%N` | node list | **compressed**: `h100-[1-2]` means `h100-1,h100-2` |
| `%P` | partition | queue |
| `%C` | CPUs | allocated to the job |
| `%m` | memory | requested — but see §5, this field is a trap |
| `%b` | GRES | requested GPUs |
| `%M`, `%L` | time | elapsed / time left |

In [8]:
!squeue -o "%i|%u|%j|%N|%P|%C|%m|%b|%M|%L"

10020_74|varun|pretrain|h100-[1-2]|h100|16|100G|gpu:2|10:00|20:00
2001|sofie|finetune|h100-2|h100|8|51200|gpu:1|1:00:00|3:00:00
3003|varun|sim|h200-1|h200|12|0|gpu:2|5:00|6:00
4004|sofie|infer|b200-1|b200|32|200G|gpu:4|3:00|4:00
5005|admin|bench|h200-2|h200|24|100G|gpu:2|2:00:00|5:00:00
6006|snooby|data-prep|cpu-1|cpu|32|256G||12:00:00|2-00:00:00
7007|admin|eval|cpu-1|cpu|8|0||4:00:00|10:00:00
8008|snooby|train-xl|b200-1|b200|16|500G|gpu:8|1:00:00|13:00:00


### Queue depth

Pending jobs are the other half of the picture. Counting them per partition
tells you how long new work will wait:

```bash
squeue -t PENDING | sort | uniq -c | sort -rn
```

In [9]:
!squeue -t PENDING | sort | uniq -c | sort -rn

   4 h100
   3 cpu
   2 h200
   1 b200


> **Why is my job pending?** Ask for the reason:
> `squeue -o "%i %R"`. Common answers: `Resources` (nothing free yet),
> `Priority` (others are ahead), `ReqNodeNotAvail` (the node you pinned is
> down or drained), and `QOSMaxJobsPerUserLimit` (you hit your account's
> limit — see §8).
>
> Note the compressed node list: one job showing `h100-[1-2]` occupies
> **two** nodes. Parse it before counting where work runs.

## 5 · `scontrol show job` — the request, exactly as stored

`squeue` is a view; `scontrol show job` is the record the scheduler will
enforce. It also settles the biggest source of confusion in Slurm
monitoring: **memory units**.

```bash
scontrol show job 10020_74
```

In [10]:
!scontrol show job 10020_74

JobId=10020_74 UserId=varun(42) GroupId=kapoorlabs(42) JobName=pretrain RunTime=10:00 TimeLimit=20:00 StartTime=2026-11-22T08:00:00 EndTime=2026-11-22T20:00:00 MinMemory=100G TresPerNode=gres:gpu:2 Command=./pretrain.sh WorkDir=/home/varun/jobs StdOut=/home/varun/jobs/10020_74.out NodeList=h100-[1-2] Partition=h100 NumCPUs=16 State=RUNNING


Here `MinMemory=100G` is a whole-node request: 100 GB for the job. A
per-CPU request looks different — this job asked for 5 GB per CPU:

```bash
scontrol show job 3003
```

In [11]:
!scontrol show job 3003

JobId=3003 UserId=varun(42) GroupId=kapoorlabs(42) JobName=sim RunTime=5:00 TimeLimit=6:00 StartTime=2026-11-22T08:00:00 EndTime=2026-11-22T20:00:00 MinCPUsMemory=5120M MinMemory=0 TresPerNode=gres:gpu:2 Command=./sim.sh WorkDir=/home/varun/jobs StdOut=/home/varun/jobs/3003.out NodeList=h200-1 Partition=h200 NumCPUs=12 State=RUNNING


| What you see | What it means |
|---|---|
| `MinMemory=100G` | `--mem 100G` — per **node** |
| `MinMemory=1024c` | legacy per-CPU form |
| `MinMemory=0 MinCPUsMemory=5G` | modern `--mem-per-cpu 5G` |
| `MinMemory=0` | the job requested **no** memory at all |

> **The `%m` trap.** `squeue %m` usually prints plain MB, and
> `int(mem)/1024` looks like it works — until someone submits `--mem 100G`
> and the parser silently reports **0 GB** for every job. Prefer
> `scontrol show job -o`, multiply per-CPU requests by the job's CPU count,
> and render "no request" as words, never as `0 GB`.

The record also carries `TresPerNode=gres:gpu:2` (GPUs per node),
`NodeList`, `RunTime`, `TimeLimit`, `Command` and `WorkDir` — everything you
need to reproduce or explain a job.

## 6 · `sstat` — live memory, from the node itself

`sstat` reports what a **running** job is actually using, measured on its
node and relayed by the controller. No SSH required.

```bash
sstat -j 10020_74,2001 --format=JobID,MaxRSS -P --noheadings
```

In [12]:
!sstat -j 10020_74,2001 --format=JobID,MaxRSS -P --noheadings

10020_74.batch|67108864K
2001.batch|5242880K


- One line **per job step**: `.batch`, `.0`, `.1`, … Take the max after
  stripping the suffix to get the job's peak.
- Units differ from the scheduler fields. `sstat` bare values are **KB**;
  `sinfo` / `squeue` bare values are **MB**. Here the values end in `K`.
- Empty output means the cluster's accounting does not collect this — show
  "unknown", never zero. `AccountingStorageTRES` in `slurm.conf` decides
  (see §3).

## 7 · `sacct` — history and measured usage

`sstat` is live only. `sacct` reads the accounting database (`slurmdbd`)
for jobs that have finished — and for running jobs too.

```bash
sacct -u varun --starttime now-24hours \
      --format=JobID,JobName,State,ExitCode,NodeList -X -P
```

- `-X` — allocation records only, skip per-step lines;
- `-P` — parsable: `|` separated, no alignment padding;
- `--starttime now-24hours` — relative windows work; pair with `--endtime`;
- `-a` — all users instead of `-u <user>`.

`State` is where you diagnose failures:

| State | What happened |
|---|---|
| `COMPLETED` | exit code 0 |
| `FAILED` | non-zero exit code — read `ExitCode` |
| `TIMEOUT` | hit the partition's time limit |
| `OUT_OF_MEMORY` | the kernel killed it — first place to look when a job "just died" |
| `CANCELLED` | `scancel`, or the user's session ended |

In [13]:
!sacct -u varun --starttime now-24hours --format=JobID,JobName,State,ExitCode,NodeList -X -P

JobID|JobName|State|ExitCode|NodeList
10020_74|pretrain|RUNNING|0:0|h100-[1-2]
3003|sim|RUNNING|0:0|h200-1
9001|pretrain|COMPLETED|0:0|h100-1
9002|pretrain|FAILED|1:0|h100-1
9008|sim|COMPLETED|0:0|h200-1


### Putting it together — `slurm_job_checker.py`

The fourth script automates the diagnosis you would otherwise do by hand:

1. list each user's jobs with `sacct` over a window;
2. pull full detail per job with `scontrol show job`;
3. flag **failed**, **stuck** (running far past a sane time) and
   **unkillable** (cancelling but still running) jobs;
4. exit non-zero when it finds problems — that exit code is the signal a
   cron job or CI check would look at.

Users and window live in `conf/job_checker.yaml`; override them on the
command line:

```bash
python slurm_job_checker.py 'users=[varun,sofie]' hours=48
```

In [14]:
!python slurm_job_checker.py 'users=[varun,sofie]' hours=24 || echo "exit 1: problems were flagged (expected)"

SLURM Job Status Checker
Users: varun, sofie
Time range: Last 24 hours

Checking user: varun
Found 5 jobs in last 24 hours


  [1/5] Checking job 10020_74... 

[RUNNING]
  [2/5] Checking job 3003... [RUNNING]
  [3/5] Checking job 9001... 

[COMPLETED]
  [4/5] Checking job 9002... 

[FAILED]
  [5/5] Checking job 9008... 

[COMPLETED]

Checking user: sofie
Found 4 jobs in last 24 hours


  [1/4] Checking job 2001... 

[RUNNING]
  [2/4] Checking job 4004... [RUNNING]
  [3/4] Checking job 9003... 

[PROBLEMATIC]
  [4/4] Checking job 9006... 

[COMPLETED]

SUMMARY
Total jobs checked: 9
  Running:      4
  Failed:       1
  Unkillable:   0
  Problematic:  1
  Completed:    3

USER: varun

FAILED JOBS (1):
----------------------------------------------------------------------

  Job ID: 9002
    Name:       pretrain
    State:      FAILED
    Exit Code:  1:0
    Node:       h100-1
    Start:      2026-11-18T09:00:00
    End:        2026-11-19T09:00:00
    Reason:     N/A

RUNNING JOBS (2):
----------------------------------------------------------------------

  Job ID: 10020_74
    Name:       pretrain
    State:      RUNNING
    Exit Code:  0:0
    Node:       h100-[1-2]
    Start:      2026-11-22T08:00:00
    End:        Unknown
    Reason:     N/A

  Job ID: 3003
    Name:       sim
    State:      RUNNING
    Exit Code:  0:0
    Node:       h200-1
    Start:      2026-11-22T08:00:00
    End:        Unknown
    Reason:     N/A

COMPLETED JOBS (2):
----------------------------------------------------------------------

  Jo

> **Read the `PROBLEMATIC` block first.** In this fake history, `9003`
> (`finetune`) hit `OUT_OF_MEMORY` — the node killed it, the job did not
> fail on its own. The fix belongs in the submission script (`--mem`), not
> in the code.
>
> `ExitCode` is the job's `exit:signal` pair, not the monitoring script's
> own status.

## 8 · `sacctmgr` — what you are allowed to run

Access in Slurm is a three-part association: **account + partition + QOS**.
`sacctmgr` prints it.

```bash
sacctmgr -nP show associations user=snooby format=Account,Partition,QOS
```

In [15]:
!sacctmgr -nP show associations user=varun format=Account,Partition,QOS
!sacctmgr -nP show associations user=snooby format=Account,Partition,QOS

kapoorlabs|ALL|normal


kapoorlabs-cpu|cpu|normal


- `Partition=ALL` — can submit to every queue;
- a list such as `cpu` — restricted to those partitions;
- `QOS=high` — allowed to jump the queue, but often capped by
  `MaxJobsPerUser` or `MaxTRESPerUser`.

> **A pending job with a `QOS*Limit*` reason is not waiting for hardware.**
> It is waiting for permission. `sinfo` will show free CPUs; `sacctmgr`
> explains why they are unusable to that user.

## 9 · Cheat sheet

| I want to know… | Command |
|---|---|
| What hardware exists | `sinfo -N -o "%N %P %C %m %G %T"` |
| Which nodes are free | `sinfo` + the free-node script (idle CPUs, not drained) |
| Where to run right now | `python show_free_nodes.py` → `srun --nodelist=… --pty bash` |
| What a node has reserved | `scontrol show node <node>` (`AllocTRES`) |
| What is running | `squeue -o "%i %u %j %N %P %C %M %L"` |
| Why my job is pending | `squeue -o "%i %R"` |
| What a job requested | `scontrol show job <jobid>` (`MinMemory`, `TresPerNode`) |
| How much memory it uses | `sstat -j <jobid> --format=JobID,MaxRSS` |
| What happened to a job | `sacct -j <jobid> -X -o JobID,State,ExitCode` |
| What I may run | `sacctmgr -nP show associations user=$USER format=Account,Partition,QOS` |
| Kill a job | `scancel <jobid>` |

**The one rule that prevents most wrong answers:** a free CPU count is not
permission to run. Check the node state (`drained`?), the user's association
(`sacctmgr`), and the job record (`scontrol show job`) before concluding
anything.

## 10 · The live dashboard — `hpc_dashboard.py`

Everything above, live, in a browser: three tabs (partitions, node detail,
top jobs) built from the same commands, with nothing hardcoded to one
cluster. It is the finished tool, not another exercise.

**Deploy it on the head node.** Start it inside `tmux` so it survives a
dropped connection:

```bash
tmux new -s dashboard
streamlit run hpc_dashboard.py        # serves on port 8501
```

**Reach it from your laptop** with an SSH tunnel:

```bash
ssh -N -L 8501:localhost:8501 <user>@<headnode>
```

Then open **http://localhost:8501**.

> **Tunnel caveat:** auto-refresh does not travel through the tunnel
> reliably. The page fetches new data when you interact with it or press
> *Refresh now*, not on a timer.

---

**Next:** [`SLURM_TUTORIAL.md`](../../SLURM_TUTORIAL.md) for the full
reference behind every command, including the memory-unit traps and the
GRES parsing rules in more detail.